# Sesión 5 · Árboles de decisión y ensembles
**IA aplicada a la ciberseguridad · Máster en Ciberseguridad**

| Parte | Qué haces |
|---|---|
| A | Un árbol desde cero: Gini y mejor corte |
| B | Afinar y leer un árbol: de modelo a regla |
| C | Random Forest, OOB e importancias |
| D | Boosting con parada temprana |
| E | Atajos: por qué un 99,9 % debe hacer sospechar |
| F | Evasión de juguete contra un bosque |

> **Dos rutas en este cuaderno.** La **ruta base** ejecuta y observa: cada parte explica qué deberías ver. Los bloques **Reto avanzado** son opcionales y piden método.
>
> Ruta base: partes A a E. Ruta avanzada: retos y parte F.
>
> Ejecuta las celdas **en orden** (`Mayús + Intro`). En Colab, si se desconecta el entorno, vuelve a ejecutar la preparación.

## Preparación

In [ ]:
# Preparación del entorno: ejecuta esta celda la primera (Colab, VS Code o Codespaces)
import sys, os, subprocess, importlib.util
EN_COLAB = "google.colab" in sys.modules
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter en VS Code / Codespaces")
paquetes = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "joblib": "joblib", "xgboost": "xgboost", "lightgbm": "lightgbm"}
faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

# Funciones del curso en ia_utils.py (no modifica tu utils.py)
IA_UTILS = "\"\"\"Funciones del curso IA aplicada a la ciberseguridad (sesiones 3 a 6).\"\"\"\nimport os, re, urllib.request\nimport numpy as np\nimport pandas as pd\n\nNSL_URL = \"https://raw.githubusercontent.com/defcom17/NSL_KDD/master/\"\nNSL_COLS = [\"duration\",\"protocol_type\",\"service\",\"flag\",\"src_bytes\",\"dst_bytes\",\"land\",\n  \"wrong_fragment\",\"urgent\",\"hot\",\"num_failed_logins\",\"logged_in\",\"num_compromised\",\n  \"root_shell\",\"su_attempted\",\"num_root\",\"num_file_creations\",\"num_shells\",\n  \"num_access_files\",\"num_outbound_cmds\",\"is_host_login\",\"is_guest_login\",\"count\",\n  \"srv_count\",\"serror_rate\",\"srv_serror_rate\",\"rerror_rate\",\"srv_rerror_rate\",\n  \"same_srv_rate\",\"diff_srv_rate\",\"srv_diff_host_rate\",\"dst_host_count\",\n  \"dst_host_srv_count\",\"dst_host_same_srv_rate\",\"dst_host_diff_srv_rate\",\n  \"dst_host_same_src_port_rate\",\"dst_host_srv_diff_host_rate\",\"dst_host_serror_rate\",\n  \"dst_host_srv_serror_rate\",\"dst_host_rerror_rate\",\"dst_host_srv_rerror_rate\",\n  \"label\",\"difficulty\"]\n\n\ndef load_nsl_kdd(split=\"train\", base=\"data\"):\n    \"\"\"Carga NSL-KDD y añade la columna is_attack. Descarga los ficheros si faltan.\"\"\"\n    name = \"KDDTrain+.txt\" if split == \"train\" else \"KDDTest+.txt\"\n    path = os.path.join(base, name)\n    if not os.path.exists(path):\n        os.makedirs(base, exist_ok=True)\n        urllib.request.urlretrieve(NSL_URL + name.replace(\"+\", \"%2B\"), path)\n    df = pd.read_csv(path, header=None, names=NSL_COLS)\n    df[\"is_attack\"] = (df[\"label\"] != \"normal\").astype(int)\n    return df\n\n\nATTACK_MAP = {\"normal\": \"normal\",\n  **{a: \"dos\" for a in [\"back\", \"land\", \"neptune\", \"pod\", \"smurf\", \"teardrop\",\n                        \"apache2\", \"udpstorm\", \"processtable\", \"mailbomb\"]},\n  **{a: \"probe\" for a in [\"satan\", \"ipsweep\", \"nmap\", \"portsweep\", \"mscan\", \"saint\"]},\n  **{a: \"r2l\" for a in [\"guess_passwd\", \"ftp_write\", \"imap\", \"phf\", \"multihop\",\n                        \"warezmaster\", \"warezclient\", \"spy\", \"xlock\", \"xsnoop\",\n                        \"snmpguess\", \"snmpgetattack\", \"httptunnel\", \"sendmail\",\n                        \"named\", \"worm\"]},\n  **{a: \"u2r\" for a in [\"buffer_overflow\", \"loadmodule\", \"rootkit\", \"perl\",\n                        \"sqlattack\", \"xterm\", \"ps\"]}}\n\n\ndef add_category(df):\n    \"\"\"Añade la columna category: normal, dos, probe, r2l o u2r.\"\"\"\n    df[\"category\"] = df[\"label\"].map(ATTACK_MAP)\n    return df\n\n\ndef generar_flujos(n=60_000, frac_ataque=0.25, seed=0, cambio_herramienta=False):\n    \"\"\"Flujos de red sintéticos con columnas al estilo CICIDS2017, incluidos sus defectos\n    (espacios en los nombres, infinitos, nulos y duplicados). Etiquetas BENIGN / DDoS.\n    Con cambio_herramienta=True el ataque llega en dos episodios con herramientas distintas.\"\"\"\n    rng = np.random.default_rng(seed)\n    nb, na = int(n * (1 - frac_ataque)), int(n * frac_ataque)\n    t0 = pd.Timestamp(\"2017-07-07 09:00:00\")\n    # --- tráfico benigno\n    port_b = rng.choice([443, 80, 53, 22, 8080, 3389, 0], nb, p=[.40, .20, .20, .05, .05, .02, .08])\n    port_b = np.where(port_b == 0, rng.integers(1024, 65535, nb), port_b)\n    dur_b = np.where(rng.random(nb) < 0.02, 0, rng.lognormal(11, 2, nb)).astype(int)\n    fwd_b = rng.poisson(8, nb) + 1\n    bwd_b = rng.poisson(7, nb)\n    fwl_b = np.where(rng.random(nb) < 0.15, rng.uniform(0, 10, nb), rng.normal(400, 250, nb)).clip(0)\n    bwl_b = rng.normal(700, 400, nb).clip(0)\n    syn_b = (rng.random(nb) < 0.3).astype(int)\n    win_b = rng.choice([29200, 65535, 8192, 14600, 64240, -1], nb, p=[.25, .25, .15, .1, .15, .1])\n    ts_b = rng.uniform(0, 8 * 3600, nb)\n    # --- ataque\n    if cambio_herramienta:\n        epi = rng.random(na) < 0.5\n        ts_a = np.where(epi, rng.uniform(1 * 3600, 2 * 3600, na), rng.uniform(5 * 3600, 6 * 3600, na))\n        port_a = np.where(epi, 80, rng.choice([8080, 443], na))\n        win_a = np.where(epi, 8192, rng.choice([29200, 65535, 64240], na))\n        fwl_a = np.where(epi, rng.normal(8, 6, na), rng.normal(60, 40, na)).clip(0)\n    else:\n        ts_a = rng.uniform(6.9 * 3600, 7.3 * 3600, na)\n        port_a = np.where(rng.random(na) < 0.9, 80, 443)\n        win_a = np.where(rng.random(na) < 0.95, 8192, 29200)\n        fwl_a = rng.normal(8, 6, na).clip(0)\n    dur_a = rng.lognormal(13, 1.5, na).astype(int)\n    fwd_a = rng.poisson(4, na) + 1\n    bwd_a = rng.poisson(3, na)\n    bwl_a = rng.normal(1500, 600, na).clip(0)\n    syn_a = (rng.random(na) < 0.6).astype(int)\n\n    def bloque(port, dur, fwd, bwd, fwl, bwl, syn, win, ts, label):\n        tot_f, tot_b = fwd * fwl, bwd * bwl\n        seg = np.maximum(dur, 0) / 1e6\n        with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n            fbs = (tot_f + tot_b) / seg\n            fps = (fwd + bwd) / seg\n            iat = np.where(fwd + bwd > 1, dur / np.maximum(fwd + bwd - 1, 1), np.nan)\n        return pd.DataFrame({\n            \" Destination Port\": port, \" Flow Duration\": dur, \" Total Fwd Packets\": fwd,\n            \" Total Backward Packets\": bwd, \"Total Length of Fwd Packets\": tot_f.round(1),\n            \" Total Length of Bwd Packets\": tot_b.round(1), \" Fwd Packet Length Mean\": fwl.round(2),\n            \" Bwd Packet Length Mean\": bwl.round(2), \"Flow Bytes/s\": fbs, \" Flow Packets/s\": fps,\n            \" Flow IAT Mean\": iat, \" SYN Flag Count\": syn, \" ACK Flag Count\": (rng.random(len(port)) < 0.85).astype(int),\n            \"Init_Win_bytes_forward\": win, \" Avg Fwd Segment Size\": fwl.round(2),\n            \" Timestamp\": t0 + pd.to_timedelta(ts, unit=\"s\"), \" Label\": label})\n\n    df = pd.concat([bloque(port_b, dur_b, fwd_b, bwd_b, fwl_b, bwl_b, syn_b, win_b, ts_b, \"BENIGN\"),\n                    bloque(port_a, dur_a, fwd_a, bwd_a, fwl_a, bwl_a, syn_a, win_a, ts_a, \"DDoS\")], ignore_index=True)\n    df.loc[rng.random(len(df)) < 0.001, \"Flow Bytes/s\"] = np.nan\n    df = pd.concat([df, df.sample(frac=0.01, random_state=seed)], ignore_index=True)\n    return df.sort_values(\" Timestamp\").reset_index(drop=True)\n\n\ndef generar_authlog(ruta=\"auth.log\", n_atacantes=12, seed=42):\n    \"\"\"auth.log sintético de la sesión 2: 40 IP legítimas y n_atacantes IP de fuerza bruta (203.0.113.0/24).\"\"\"\n    import datetime as dt\n    rng = np.random.default_rng(seed); start = dt.datetime(2026, 10, 1)\n    users = [\"root\", \"admin\", \"carlos\", \"ana\", \"luis\", \"deploy\", \"test\", \"oracle\", \"ubuntu\", \"postgres\"]\n    rows = []\n    for i in range(40):\n        ip = f\"10.10.10.{10 + i}\"\n        for t in rng.uniform(0, 7 * 24 * 3600, rng.integers(5, 40)):\n            rows.append((start + dt.timedelta(seconds=float(t)), bool(rng.random() > 0.08), str(rng.choice(users[2:6])), ip))\n    for i in range(n_atacantes):\n        ip = f\"203.0.113.{20 + i}\"; t0 = rng.uniform(0, 7 * 24 * 3600 - 3600)\n        for t in t0 + np.sort(rng.uniform(0, 1800, rng.integers(60, 400))):\n            rows.append((start + dt.timedelta(seconds=float(t)), False, str(rng.choice(users)), ip))\n    rows.sort(key=lambda r: r[0])\n    with open(ruta, \"w\") as f:\n        for ts, ok, user, ip in rows:\n            v = \"Accepted\" if ok else \"Failed\"\n            f.write(f\"{ts:%b %d %H:%M:%S} srv sshd[{rng.integers(1000, 9999)}]: {v} password \"\n                    f\"for {user} from {ip} port {rng.integers(1024, 65000)} ssh2\\n\")\n    return ruta\n\n\ndef auth_features(ruta=\"auth.log\"):\n    \"\"\"Características por IP y ventana de 10 minutos. La columna verdad solo sirve para evaluar.\"\"\"\n    pat = re.compile(r\"^(\\w{3}\\s+\\d+ \\d\\d:\\d\\d:\\d\\d) \\S+ sshd\\[\\d+\\]: (Failed|Accepted) password for (\\S+) from (\\S+) port\")\n    reg = []\n    for linea in open(ruta):\n        m = pat.match(linea)\n        if m:\n            reg.append((pd.to_datetime(\"2026 \" + m.group(1), format=\"%Y %b %d %H:%M:%S\"),\n                        m.group(2) == \"Accepted\", m.group(3), m.group(4)))\n    log = pd.DataFrame(reg, columns=[\"ts\", \"ok\", \"user\", \"ip\"])\n    log[\"win\"] = log[\"ts\"].dt.floor(\"10min\")\n    f = log.groupby([\"ip\", \"win\"]).agg(n=(\"ok\", \"size\"), n_ok=(\"ok\", \"sum\"), n_users=(\"user\", \"nunique\")).reset_index()\n    f[\"n_fail\"] = f[\"n\"] - f[\"n_ok\"]\n    f[\"fail_ratio\"] = f[\"n_fail\"] / f[\"n\"]\n    f[\"verdad\"] = f[\"ip\"].str.startswith(\"203.0.113.\").astype(int)\n    return f\n"
with open("ia_utils.py", "w", encoding="utf-8") as f:
    f.write(IA_UTILS)

import importlib, ia_utils; importlib.reload(ia_utils)
from ia_utils import *
tr, te = load_nsl_kdd("train"), load_nsl_kdd("test")      # descarga NSL-KDD si falta
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
plt.rcParams["figure.figsize"] = (9, 4)
print("NSL-KDD:", tr.shape, te.shape, "| scikit-learn", sklearn.__version__)

## Parte A · Un árbol desde cero
**Gini** mide cuánta mezcla hay en un grupo: 0 si es puro, 0,5 si está al 50 %. Un árbol busca el corte que **más reduce la mezcla**.

In [ ]:
def gini(y):
    if len(y) == 0: return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - (p ** 2).sum()

# Ejemplo de la teoría: 6 ataques y 4 normales
padre = np.array([1] * 6 + [0] * 4)
izq, der = np.array([1] * 5), np.array([1] + [0] * 4)
g_hijos = (len(izq) * gini(izq) + len(der) * gini(der)) / len(padre)
print(f"Gini padre {gini(padre):.2f} | hijos ponderado {g_hijos:.2f} | ganancia {gini(padre) - g_hijos:.2f}")

In [ ]:
def mejor_corte(x, y):
    o = np.argsort(x); xs, ys = x[o], y[o]; mejor = (None, np.inf)
    for i in range(1, len(xs)):
        if xs[i] == xs[i - 1]: continue
        g = (i * gini(ys[:i]) + (len(ys) - i) * gini(ys[i:])) / len(ys)
        if g < mejor[1]: mejor = ((xs[i] + xs[i - 1]) / 2, g)
    return mejor
from sklearn.tree import DecisionTreeClassifier
muestra = tr.sample(3000, random_state=0); y_m = muestra.is_attack.to_numpy()
for col in ["serror_rate", "same_srv_rate", "count", "src_bytes"]:
    t, g = mejor_corte(muestra[col].to_numpy(), y_m)
    t_sk = DecisionTreeClassifier(max_depth=1).fit(muestra[[col]], y_m).tree_.threshold[0]
    print(f"{col:14s} corte propio={t:10.3f}  scikit-learn={t_sk:10.3f}  Gini={g:.3f}")

**Qué deberías ver:** Gini padre 0,48, hijos 0,16, ganancia 0,32; y los cortes propios coinciden con los de scikit-learn.

**Reto avanzado.** Extiende `mejor_corte` para devolver la ganancia respecto al Gini del padre y ordena todas las columnas numéricas por ganancia. Esa es la primera pregunta que haría el árbol.

In [ ]:
# Tu código

## Parte B · Afinar y leer un árbol
La validación sale **del train**: el test no se toca hasta el final.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import export_text, plot_tree
from sklearn.metrics import classification_report, average_precision_score
num = ["duration", "src_bytes", "dst_bytes", "count", "srv_count", "serror_rate", "rerror_rate", "same_srv_rate",
       "diff_srv_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_serror_rate",
       "logged_in", "hot", "num_failed_logins", "num_compromised", "root_shell", "num_root"]
X, y = tr[num], tr.is_attack
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    {"max_depth": [3, 6, 10, None], "min_samples_leaf": [1, 20, 100]},
                    cv=StratifiedKFold(3, shuffle=True, random_state=0), scoring="average_precision", n_jobs=-1).fit(X, y)
print(grid.best_params_, round(grid.best_score_, 4))
mini = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=0).fit(X, y)
print(export_text(mini, feature_names=num))
plt.figure(figsize=(16, 6)); plot_tree(mini, feature_names=num, class_names=["normal", "ataque"], filled=True, fontsize=8); plt.show()
print("test · árbol afinado:", round(grid.best_estimator_.score(te[num], te.is_attack), 3), "| mini árbol:", round(mini.score(te[num], te.is_attack), 3))

**Qué deberías ver:** un árbol de 3 niveles legible como reglas «si… entonces…», que pierde poca exactitud frente al afinado.

**Reto avanzado.** Escribe las tres reglas más importantes en lenguaje natural y propón cómo serían en una regla Sigma. ¿Por qué no sirven tal cual en una red moderna?

In [ ]:
# Tu código

## Parte C · Random Forest, OOB e importancias

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, oob_score=True, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=0).fit(Xtr, ytr)
print(f"OOB {rf.oob_score_:.4f} | validación {rf.score(Xval, yval):.4f} | test {rf.score(te[num], te.is_attack):.4f}")
res = [(n, RandomForestClassifier(n_estimators=n, min_samples_leaf=2, n_jobs=-1, random_state=0).fit(Xtr, ytr).score(Xval, yval))
       for n in [1, 5, 25, 100]]
pd.DataFrame(res, columns=["árboles", "acc_val"]).set_index("árboles").plot(marker="o", title="Más árboles, menos varianza"); plt.show()

In [ ]:
vs = Xval.sample(5000, random_state=0); vy = yval.loc[vs.index]
perm = permutation_importance(rf, vs, vy, n_repeats=3, random_state=0, n_jobs=-1, scoring="average_precision")
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
pd.Series(rf.feature_importances_, index=num).sort_values().plot.barh(ax=ax[0], title="Importancia por impureza")
pd.Series(perm.importances_mean, index=num).sort_values().plot.barh(ax=ax[1], title="Importancia por permutación")
plt.tight_layout(); plt.show()

**Qué deberías ver:** OOB y validación casi iguales, test más bajo (NSL-KDD test trae ataques nuevos) y dos rankings de importancia que no coinciden del todo.

**Reto avanzado.** Mide el recall por categoría en el test con `add_category`. Calcula la varianza de un promedio de B = 100 árboles con σ² = 1 y ρ = 0,3: ρσ² + (1 − ρ)σ²/B. ¿Por qué no baja de 0,3 aunque añadas árboles?

In [ ]:
# Tu código

## Parte D · Boosting con parada temprana

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
import time
filas = []
for nom, m in {"HistGB": HistGradientBoostingClassifier(max_iter=500, early_stopping=True, validation_fraction=0.15,
                                                        n_iter_no_change=20, random_state=0),
               "XGBoost": XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                        eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)}.items():
    t0 = time.perf_counter()
    if nom == "XGBoost": m.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False); rondas = m.best_iteration
    else: m.fit(Xtr, ytr); rondas = m.n_iter_
    pr = m.predict_proba(te[num])[:, 1]
    filas.append({"modelo": nom, "rondas": rondas, "t_fit_s": round(time.perf_counter() - t0, 1),
                  "AP_test": average_precision_score(te.is_attack, pr), "acc_test": ((pr > 0.5) == te.is_attack).mean()})
    if nom == "XGBoost": xgb = m
pr = rf.predict_proba(te[num])[:, 1]
filas.append({"modelo": "RandomForest", "rondas": 200, "t_fit_s": None, "AP_test": average_precision_score(te.is_attack, pr),
              "acc_test": rf.score(te[num], te.is_attack)})
print(pd.DataFrame(filas).round(3))
plt.plot(xgb.evals_result()["validation_0"]["logloss"]); plt.axvline(xgb.best_iteration, color="red", ls="--")
plt.title("XGBoost: log-loss en validación y ronda de parada"); plt.show()

**Reto avanzado.** Cambia `learning_rate` a 0,01 y 0,5: ¿cuántas rondas necesita cada uno y cuánto rinde? Prueba `scale_pos_weight` en un problema u2r frente al resto. ¿Qué modelo desplegarías en un IDS y por qué?

In [ ]:
# Tu código

## Parte E · Atajos: el 99,9 % sospechoso
Flujos sintéticos con **dos episodios de ataque lanzados con herramientas distintas**. La primera deja una huella de laboratorio: `Init_Win_bytes_forward = 8192` y puerto 80.

In [ ]:
w = generar_flujos(seed=1, cambio_herramienta=True)
w.columns = w.columns.str.strip()
w = w.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates().reset_index(drop=True)
w["y"] = (w["Label"] != "BENIGN").astype(int)
cols = [c for c in w.columns if c not in ("Label", "y", "Timestamp")]
# 1) división aleatoria
Xa, Xb, ya, yb = train_test_split(w[cols], w.y, test_size=0.3, stratify=w.y, random_state=0)
rf1 = RandomForestClassifier(100, n_jobs=-1, random_state=0).fit(Xa, ya)
print("aleatoria, todas las columnas:", round(rf1.score(Xb, yb), 4))
print(pd.Series(rf1.feature_importances_, index=cols).sort_values(ascending=False).head(5).round(3))
# 2) división temporal: mañana para entrenar, tarde para probar
corte = pd.Timestamp("2017-07-07 12:30")
ant, des = w[w.Timestamp < corte], w[w.Timestamp >= corte]
rf2 = RandomForestClassifier(100, n_jobs=-1, random_state=0).fit(ant[cols], ant.y)
print("temporal, todas las columnas:", round(rf2.score(des[cols], des.y), 4), "| recall:", round(des.y[rf2.predict(des[cols]) == 1].count() / des.y.sum(), 3))

**Qué deberías ver:** casi un 99 % con división aleatoria; con división temporal el recall del segundo episodio se hunde casi a cero. **El modelo había aprendido la huella de la primera herramienta, no el comportamiento del ataque.**

**Reto avanzado.** Ablación: elimina `Destination Port` e `Init_Win_bytes_forward` y repite las dos divisiones. ¿Qué características comportamentales quedan y cuánto recuperas? Redacta en tres líneas el riesgo de publicar un 99,9 % sin esta comprobación.

In [ ]:
# Tu código

## Parte F · Evasión de juguete
Solo con fines didácticos y sobre datos públicos: ¿cuántos ataques detectados dejan de serlo si el atacante cambia **una** característica que controla?

In [ ]:
controlables = ["duration", "src_bytes", "dst_bytes", "count", "srv_count"]
atq = te[te.is_attack == 1].sample(500, random_state=0)
det = atq[rf.predict(atq[num]) == 1]
evadido = pd.Series(False, index=det.index)
for col in controlables:
    for factor in [0.1, 0.5, 2, 10]:
        cand = det[num].astype(float).copy(); cand[col] = cand[col] * factor
        evadido |= rf.predict(cand) == 0
print(f"{evadido.sum()}/{len(det)} ataques detectados se evaden tocando una sola característica controlable")

**Reto avanzado.** Propón contramedidas: entrenar solo con características que el atacante no controla, entrenamiento adversarial, varios modelos y detección de anomalías complementaria (sesión 6).

In [ ]:
# Tu código

## Entrega
Reinicia y ejecuta todo. Entrega `ia5_arboles.ipynb` con la tabla Random Forest frente a boosting y tu conclusión sobre los atajos de la parte E.